# 01_Data_Inspection

## Objective

The goal of this notebook is to perform an initial exploratory inspection of the core MIMIC-III tables required for the MRP project.

This notebook focuses on:

- loading the main datasets (`ADMISSIONS`, `PATIENTS`, `ICUSTAYS`, `DIAGNOSES_ICD`, and `NOTEEVENTS`)
- examining dataset shapes (rows and columns)
- reviewing column names and table structure
- checking missing values across key variables
- previewing sample records from each table
- identifying the most important columns for cohort creation and modeling
- understanding how the tables will be linked using patient and admission identifiers

The final outcome of this notebook is a clear understanding of the available data and readiness to proceed to cohort creation in the next stage of the project.

In [1]:
# Code Starter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"
OUTPUT_PATH = "../outputs/"

### 1. ) Load Core Tables
Used sample rows for notes first because it’s huge

In [5]:
admissions = pd.read_csv(DATA_PATH + "ADMISSIONS.csv")
patients = pd.read_csv(DATA_PATH + "PATIENTS.csv")
icustays = pd.read_csv(DATA_PATH + "ICUSTAYS.csv")
diagnoses = pd.read_csv(DATA_PATH + "DIAGNOSES_ICD.csv")
notes = pd.read_csv(DATA_PATH + "NOTEEVENTS.csv", nrows=50000)

### 2.) Inspect Shape of Tables to confirm size

In [9]:
print(admissions.shape)
print(patients.shape)
print(icustays.shape)
print(diagnoses.shape)
print(notes.shape)

(58976, 19)
(46520, 8)
(61532, 12)
(651047, 5)
(50000, 11)


### 3.) Check Columns to confirm column names

In [18]:
tables = {
    "ADMISSIONS": admissions,
    "PATIENTS": patients,
    "ICUSTAYS": icustays,
    "DIAGNOSES_ICD": diagnoses,
    "NOTEEVENTS": notes
}

for name, df in tables.items():
    print(f"\n{name}")
    print(df.columns.tolist())


ADMISSIONS
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'DISCHARGE_LOCATION', 'INSURANCE', 'LANGUAGE', 'RELIGION', 'MARITAL_STATUS', 'ETHNICITY', 'EDREGTIME', 'EDOUTTIME', 'DIAGNOSIS', 'HOSPITAL_EXPIRE_FLAG', 'HAS_CHARTEVENTS_DATA']

PATIENTS
['ROW_ID', 'SUBJECT_ID', 'GENDER', 'DOB', 'DOD', 'DOD_HOSP', 'DOD_SSN', 'EXPIRE_FLAG']

ICUSTAYS
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'DBSOURCE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'FIRST_WARDID', 'LAST_WARDID', 'INTIME', 'OUTTIME', 'LOS']

DIAGNOSES_ICD
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'SEQ_NUM', 'ICD9_CODE']

NOTEEVENTS
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'CHARTDATE', 'CHARTTIME', 'STORETIME', 'CATEGORY', 'DESCRIPTION', 'CGID', 'ISERROR', 'TEXT']


### 4.) Missing Values for ALL Tables

In [23]:
tables = {
    "ADMISSIONS": admissions,
    "PATIENTS": patients,
    "ICUSTAYS": icustays,
    "DIAGNOSES_ICD": diagnoses,
    "NOTEEVENTS": notes
}

for name, df in tables.items():
    print("="*60)
    print(name)
    print(df.isnull().sum().sort_values(ascending=False).head(15))
    print()

ADMISSIONS
DEATHTIME               53122
EDOUTTIME               28099
EDREGTIME               28099
LANGUAGE                25332
MARITAL_STATUS          10128
RELIGION                  458
DIAGNOSIS                  25
ROW_ID                      0
HOSPITAL_EXPIRE_FLAG        0
ETHNICITY                   0
INSURANCE                   0
SUBJECT_ID                  0
DISCHARGE_LOCATION          0
ADMISSION_LOCATION          0
ADMISSION_TYPE              0
dtype: int64

PATIENTS
DOD_HOSP       36546
DOD_SSN        33142
DOD            30761
ROW_ID             0
SUBJECT_ID         0
GENDER             0
DOB                0
EXPIRE_FLAG        0
dtype: int64

ICUSTAYS
OUTTIME           10
LOS               10
ROW_ID             0
SUBJECT_ID         0
HADM_ID            0
ICUSTAY_ID         0
DBSOURCE           0
FIRST_CAREUNIT     0
LAST_CAREUNIT      0
FIRST_WARDID       0
LAST_WARDID        0
INTIME             0
dtype: int64

DIAGNOSES_ICD
SEQ_NUM       47
ICD9_CODE     47
ROW_ID     

### Preview Data Tables

In [32]:
print("ADMISSIONS")
display(admissions.head())

print("PATIENTS")
display(patients.head())

print("ICUSTAYS")
display(icustays.head())

print("DIAGNOSES_ICD")
display(diagnoses.head())

print("NOTEEVENTS")
display(notes[['SUBJECT_ID','HADM_ID','CHARTDATE','CATEGORY','TEXT']].head())

ADMISSIONS


,ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA
0,21,22,165315,2196-04-09 12:26:00,2196-04-10 15:54:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,DISC-TRAN CANCER/CHLDRN H,Private,NaN,UNOBTAINABLE,MARRIED,WHITE,2196-04-09 10:06:00,2196-04-09 13:24:00,BENZODIAZEPINE OVERDOSE,0,1
1,22,23,152223,2153-09-03 07:15:00,2153-09-08 19:10:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,MARRIED,WHITE,NaN,NaN,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1
2,23,23,124321,2157-10-18 19:34:00,2157-10-25 14:00:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,NaN,NaN,BRAIN MASS,0,1
3,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,NaN,PROTESTANT QUAKER,SINGLE,WHITE,NaN,NaN,INTERIOR MYOCARDIAL INFARCTION,0,1
4,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,NaN,UNOBTAINABLE,MARRIED,WHITE,2160-11-02 01:01:00,2160-11-02 04:27:00,ACUTE CORONARY SYNDROME,0,1


PATIENTS


,ROW_ID,SUBJECT_ID,GENDER,DOB,DOD,DOD_HOSP,DOD_SSN,EXPIRE_FLAG
0,234,249,F,2075-03-13 00:00:00,NaN,NaN,NaN,0
1,235,250,F,2164-12-27 00:00:00,2188-11-22 00:00:00,2188-11-22 00:00:00,NaN,1
2,236,251,M,2090-03-15 00:00:00,NaN,NaN,NaN,0
3,237,252,M,2078-03-06 00:00:00,NaN,NaN,NaN,0
4,238,253,F,2089-11-26 00:00:00,NaN,NaN,NaN,0


ICUSTAYS


,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,DBSOURCE,FIRST_CAREUNIT,LAST_CAREUNIT,FIRST_WARDID,LAST_WARDID,INTIME,OUTTIME,LOS
0,365,268,110404,280836,carevue,MICU,MICU,52,52,2198-02-14 23:27:38,2198-02-18 05:26:11,3.2490
1,366,269,106296,206613,carevue,MICU,MICU,52,52,2170-11-05 11:05:29,2170-11-08 17:46:57,3.2788
2,367,270,188028,220345,carevue,CCU,CCU,57,57,2128-06-24 15:05:20,2128-06-27 12:32:29,2.8939
3,368,271,173727,249196,carevue,MICU,SICU,52,23,2120-08-07 23:12:42,2120-08-10 00:39:04,2.0600
4,369,272,164716,210407,carevue,CCU,CCU,57,57,2186-12-25 21:08:04,2186-12-27 12:01:13,1.6202


DIAGNOSES_ICD


,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,1297,109,172335,1.0,40301
1,1298,109,172335,2.0,486
2,1299,109,172335,3.0,58281
3,1300,109,172335,4.0,5855
4,1301,109,172335,5.0,4254


NOTEEVENTS


,SUBJECT_ID,HADM_ID,CHARTDATE,CATEGORY,TEXT
0,22532,167853,2151-08-04,Discharge summary,Admission Date: [**2151-7-16**] Dischar...
1,13702,107527,2118-06-14,Discharge summary,Admission Date: [**2118-6-2**] Discharg...
2,13702,167118,2119-05-25,Discharge summary,Admission Date: [**2119-5-4**] D...
3,13702,196489,2124-08-18,Discharge summary,Admission Date: [**2124-7-21**] ...
4,26880,135453,2162-03-25,Discharge summary,Admission Date: [**2162-3-3**] D...


NOTE : Initial exploratory data analysis showed generally high completeness across core tables. Missingness was primarily concentrated in mortality-related fields and note timestamp metadata, while critical identifiers and note text were complete.

### Important Columns for MRP Project

### 1. ADMISSIONS

Contains hospital admission information.

| Column               | Purpose                                                 |
| -------------------- | ------------------------------------------------------- |
| SUBJECT_ID           | Unique patient ID used to connect tables                |
| HADM_ID              | Unique hospital admission ID used to connect tables     |
| ADMITTIME            | Date/time patient was admitted to hospital              |
| DISCHTIME            | Date/time patient was discharged                        |
| DEATHTIME            | Date/time patient died during admission (if applicable) |
| HOSPITAL_EXPIRE_FLAG | 1 = died in hospital, 0 = survived                      |
| DIAGNOSIS            | Main diagnosis recorded at admission                    |


### 2. PATIENTS

Contains patient demographic information.

| Column      | Purpose                             |
| ----------- | ----------------------------------- |
| SUBJECT_ID  | Unique patient ID                   |
| GENDER      | Patient gender                      |
| DOB         | Date of birth used to calculate age |
| DOD         | Date of death (if applicable)       |
| EXPIRE_FLAG | 1 = deceased, 0 = alive             |


### 3. ICUSTAYS

Contains ICU stay details.

| Column         | Purpose                              |
| -------------- | ------------------------------------ |
| SUBJECT_ID     | Unique patient ID                    |
| HADM_ID        | Links ICU stay to hospital admission |
| ICUSTAY_ID     | Unique ICU stay ID                   |
| INTIME         | Time patient entered ICU             |
| OUTTIME        | Time patient left ICU                |
| LOS            | Length of ICU stay in days           |
| FIRST_CAREUNIT | First ICU unit patient entered       |
| LAST_CAREUNIT  | Last ICU unit patient stayed in      |

Important Note: INTIME will be used to select notes written within the first 24 hours of ICU admission.


### 4. DIAGNOSES_ICD

Contains diagnosis codes.

| Column     | Purpose                                      |
| ---------- | -------------------------------------------- |
| SUBJECT_ID | Unique patient ID                            |
| HADM_ID    | Hospital admission ID                        |
| ICD9_CODE  | Diagnosis code used to identify sepsis cases |
| SEQ_NUM    | Order of diagnosis (1 = primary diagnosis)   |

Important Note: Sepsis codes such as 99591, 99592, 78552, and 038xx will be used to create the target label:

- 1 = Sepsis
- 0 = Non-Sepsis

### 5. NOTEEVENTS

Contains clinical notes (main NLP dataset).

| Column     | Purpose                                 |
| ---------- | --------------------------------------- |
| SUBJECT_ID | Unique patient ID                       |
| HADM_ID    | Hospital admission ID                   |
| CHARTDATE  | Date note was written                   |
| CHARTTIME  | Exact time note was written             |
| CATEGORY   | Type of note (Nursing, Physician, etc.) |
| ISERROR    | Indicates incorrect/error note          |
| TEXT       | Full clinical note text                 |


Most Important Column: TEXT
This is the main text data that will be used for machine learning and NLP modeling.

### PROJECT WORKFLOW SUMMARY

- PATIENTS        -> calculate age
- ICUSTAYS        -> identify ICU timing
- DIAGNOSES_ICD   -> create sepsis labels
- NOTEEVENTS      -> extract clinical text notes
- ADMISSIONS      -> additional admission information